<div align="center"> 

---



# **Cerebellum Mega Analysis**


<img src="https://static.wixstatic.com/media/da4857_5f6dbdb6bbd84328b1e31c7f2dafb673~mv2.png/v1/fill/w_224,h_226,al_c,q_85,usm_0.66_1.00_0.01,enc_auto/enigma_addiction-300x249.png" alt="image" style="width:300px;"/>


 </img>
</div>


---
# <u> **Project Description:**</u>

The aim of this study is to perform a multi-site mega/meta analysis to asses the effects of substance use on volumetric changes in cerebellar regions (and far beyond). 

## **Hypothesis to test**

**Variables to consider:** 
  - Age
  - Sex
  - Polysubstance use : Smoke and alcohol
  
  
  **Analysis:**
   1. Volume comparisons between-groups of:
   - cerebellum regions

2. Main effects: Acapulco atlas
- Cerebellar volume ~ any drug
- Cerebellar volume ~ drug type

3. Interaction effects: Cerebellum volume
- group * sex
- group + age + sex

4. Partial Least Square: Cerebral - cerebellar association
- MDTB atlas

---

In [1]:
addTaskCallback(function(...) {set.seed(42);TRUE})
options(warn=-1) # to turn off warnings
#options(warn=0) # to turn on warnings
options(repr.plot.width=10, repr.plot.height=6) # to resize plots

1 
1

In [2]:
setwd("/mnt/g/Other computers/My Laptop/PhD/Psilantro/Enigma_cerebellum/Enigma_Cerebellum/")

## Load Packages

In [3]:
# Load required packages  
  pacman::p_load(tidyverse,devtools,readxl,ggbeeswarm,gtExtras,gt,webshot,webshot2,VIM,
                ggpubr,cowplot,gtsummary,ff,ggthemes,scales,mice,vtreat,sjmisc,SmartEDA,boot,MKinfer,MatchIt,
                ggcorrplot,olsrr,lme4,rstatix,janitor,coin,rcartocolor,ggdist,gghalves,ggnewscale,patchwork,
                lsr,tidyr,reshape2,broom,irr,ggpubr,caret,tidymodels,foreach,doParallel,optmatch,emmeans)


## Set settings for plotting

In [4]:
theme_output  <- theme(text = element_text(family = "Times New Roman"),
                        axis.text.x = element_text(size = 20),
                        axis.text.y = element_text(size = 20),
                        axis.title.x = element_text(size = 25, face = "bold"),
                        axis.title.y = element_text(size = 25, face = "bold"),
                        legend.text = element_text(size = 22), legend.title = element_text(size = 24),
                        title = element_text(size = 20,face = "italic"))

pal_sex <- c(alpha("#8E5A9C",1),alpha("#2879BC",1))
pal_user <- c(alpha("#757575",1),alpha("#059DC7",1))
pal_substances <- c("Users"= "#059DC7", "HC" = alpha("#757575",1), "ALC" = alpha("#C93F4D",1), 
    "ATS" = alpha("#EB8F33",1), "CANN" = alpha("#0B8755",1), 
    "COC" = alpha("#3A5589",1), "NIC" = alpha("#764EA1",1),
    "OPI" = alpha("#06AED5",1))

In [5]:
`%notin%` <- Negate(`%in%`)

## Load data

In [6]:
load("Data/RData/Volumes_Cerebellum.RData")
load("Statistics/Linear_models/Acapulco/Acapulco_volumes_statistics.RData")

In [7]:
Acapulco_user_merged_full_folds <- merge(p_values_Acapulco_user_vs_control,
                    p_values_Acapulco_folds_users_vs_control_median,
                    by = c("region"),
                    suffixes = c("", "_folds"))

Acapulco_user_merged_full_folds_matchit <- merge(Acapulco_user_merged_full_folds,
                    p_values_Acapulco_matchit_users_vs_control,
                    by = c("region"),
                    suffixes = c("", "_matchit")) %>%
    mutate(diff_effect.size_std_FullvsMatchit = (effect.size - effect.size_matchit) / sd(effect.size - effect.size_matchit, na.rm = TRUE),
        diff_statistic_abs_FullvsMatchit = abs(statistic - statistic_matchit),
        significant_both_FullvsMatchit = ifelse(p.adj < 0.05 & p.adj_matchit < 0.05, TRUE, FALSE),
        diff_effect.size_std_FullvsFolds = (effect.size - effect.size_median) / sd(effect.size - effect.size_median, na.rm = TRUE),
        diff_statistic_abs_FullvsFolds = abs(statistic - statistic_median),
        significant_both_FullvsFolds = ifelse(p.adj < 0.05 & p_adj_median < 0.05, TRUE, FALSE),
        significant_all = ifelse(significant_both_FullvsMatchit == TRUE & significant_both_FullvsFolds == TRUE, TRUE, FALSE)
        )

Acapulco_substance_merged_full_folds <- merge(p_values_Acapulco_substance_vs_control,
                    p_values_Acapulco_folds_substance_vs_control_median,
                    by = c("region", "Substance"),
                    suffixes = c("", "_folds"))

Acapulco_substance_merged_full_folds_matchit <- merge(Acapulco_substance_merged_full_folds,
                    p_values_Acapulco_matchit_substance_vs_control,
                    by = c("region", "Substance"),
                    suffixes = c("", "_matchit")) %>%
    mutate(diff_effect.size_std_FullvsMatchit = (effect.size - effect.size_matchit) / sd(effect.size - effect.size_matchit, na.rm = TRUE),
        diff_statistic_abs_FullvsMatchit = abs(statistic - statistic_matchit),
        significant_both_FullvsMatchit = ifelse(p_adj < 0.05 & p_adj_matchit < 0.05, TRUE, FALSE),
        diff_effect.size_std_FullvsFolds = (effect.size - effect.size_median) / sd(effect.size - effect.size_median, na.rm = TRUE),
        diff_statistic_abs_FullvsFolds = abs(statistic - statistic_median),
        significant_both_FullvsFolds = ifelse(p_adj < 0.05 & p_adj_median < 0.05, TRUE, FALSE),
        significant_all = ifelse(significant_both_FullvsMatchit == TRUE & significant_both_FullvsFolds == TRUE, TRUE, FALSE)
        )

In [8]:
# ── Load CC / AI sanity-check statistics ─────────────────────────────────────
if (!exists("p_values_Acapulco_CC_user_vs_control")) {
    load("Statistics/Sanity_check/Acapulco/Acapulco_sanity_check_statistics.RData")
}
if (!exists("p_values_MDTB_CC_user_vs_control")) {
    load("Statistics/Sanity_check/MDTB/MDTB_sanity_check_statistics.RData")
}

# ── Extend significant_all to require significance across ALL 5 analyses ──────
# (Full sample + cross-validated folds + Matchit + Complete cases + Alt. imputation)

# Acapulco User vs HC: join on region only
Acapulco_user_merged_full_folds_matchit <- Acapulco_user_merged_full_folds_matchit %>%
    left_join(
        p_values_Acapulco_CC_user_vs_control %>%
            select(region, statistic, p.adj, effect.size) %>% rename(p.adj_CC = p.adj, statistic_CC = statistic, effect.size_CC = effect.size),
        by = "region"
    ) %>%
    left_join(
        p_values_Acapulco_AI_user_vs_control %>%
            select(region, statistic, p.adj, effect.size) %>% rename(p.adj_AI = p.adj, statistic_AI = statistic, effect.size_AI = effect.size),
        by = "region"
    ) %>%
    mutate(
        significant_all = significant_all &
            !is.na(p.adj_CC) & p.adj_CC < 0.05 &
            !is.na(p.adj_AI) & p.adj_AI < 0.05
    )

# Acapulco Substance vs HC: join on region AND Substance
Acapulco_substance_merged_full_folds_matchit <- Acapulco_substance_merged_full_folds_matchit %>%
    left_join(
        p_values_Acapulco_CC_substance_vs_control %>%
            select(region, Substance, statistic, p.adj, effect.size) %>% rename(p.adj_CC = p.adj, statistic_CC = statistic, effect.size_CC = effect.size),
        by = c("region", "Substance")
    ) %>%
    left_join(
        p_values_Acapulco_AI_substance_vs_control %>%
            select(region, Substance, statistic, p.adj, effect.size) %>% rename(p.adj_AI = p.adj, statistic_AI = statistic, effect.size_AI = effect.size),
        by = c("region", "Substance")
    ) %>%
    mutate(
        significant_all = significant_all &
            !is.na(p.adj_CC) & p.adj_CC < 0.05 &
            !is.na(p.adj_AI) & p.adj_AI < 0.05
    )

In [9]:
# ── Load CA sanity-check statistics ──────────────────────────────────────────
if (!exists("p_values_Acapulco_CA_user_vs_control")) {
    load("Statistics/Sanity_check/Acapulco/Acapulco_sanity_check_statistics.RData")
}

# ── Extend significant_all to require significance across ALL 6 analyses ──────
# (Full sample + cross-validated folds + Matchit + Complete cases + Alt. imputation + Covariate-Adjusted)

# Acapulco User vs HC: add CA
Acapulco_user_merged_full_folds_matchit <- Acapulco_user_merged_full_folds_matchit %>%
    left_join(
        p_values_Acapulco_CA_user_vs_control %>%
            select(region, statistic, p.adj, effect.size) %>% rename(p.adj_CA = p.adj, statistic_CA = statistic, effect.size_CA = effect.size),
        by = "region"
    ) %>%
    mutate(
        significant_all = significant_all &
            !is.na(p.adj_CA) & p.adj_CA < 0.05
    )

# Acapulco Substance vs HC: add CA
Acapulco_substance_merged_full_folds_matchit <- Acapulco_substance_merged_full_folds_matchit %>%
    left_join(
        p_values_Acapulco_CA_substance_vs_control %>%
            select(region, Substance, statistic, p.adj, effect.size) %>% rename(p.adj_CA = p.adj, statistic_CA = statistic, effect.size_CA = effect.size),
        by = c("region", "Substance")
    ) %>%
    mutate(
        significant_all = significant_all &
            !is.na(p.adj_CA) & p.adj_CA < 0.05
    )


## **1. Interclass correlation coefficient (ICC)**

Calculating the ICC between the full analysis and the median fold

### **1.1 User**

final files:  
**Acapulco_user_merged_full_folds_matchit**  
**Acapulco_user_icc**

### Calculate

#### Full vs matchID

In [10]:
Acapulco_user_icc_t_value_full_vs_matchit <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic", "statistic_matchit")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_full_vs_matchit <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size", "effect.size_matchit")], model = "twoway", type = "agreement", unit = "single")

#### Full vs folds

In [11]:
Acapulco_user_icc_t_value_full_vs_folds <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_full_vs_folds <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### Full vs CC

In [12]:
Acapulco_user_icc_t_value_full_vs_cc <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic", "statistic_CC")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_full_vs_cc <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size", "effect.size_CC")], model = "twoway", type = "agreement", unit = "single")

#### Full vs AI

In [13]:
Acapulco_user_icc_t_value_full_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_full_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### Full vs CA

In [14]:
Acapulco_user_icc_t_value_full_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_full_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs folds

In [15]:
Acapulco_user_icc_t_value_matchit_vs_folds <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_matchit", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_matchit_vs_folds <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs CC

In [16]:
Acapulco_user_icc_t_value_matchit_vs_cc <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_matchit", "statistic_CC")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_matchit_vs_cc <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_CC")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs AI

In [17]:
Acapulco_user_icc_t_value_matchit_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_matchit", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_matchit_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs CA

In [18]:
Acapulco_user_icc_t_value_matchit_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_matchit", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_matchit_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

#### folds vs CC

In [19]:
Acapulco_user_icc_t_value_folds_vs_cc <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_median", "statistic_CC")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_folds_vs_cc <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_median", "effect.size_CC")], model = "twoway", type = "agreement", unit = "single")

#### folds vs AI

In [20]:
Acapulco_user_icc_t_value_folds_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_median", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_folds_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_median", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### folds vs CA

In [21]:
Acapulco_user_icc_t_value_folds_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_median", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_folds_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_median", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

#### CC vs AI

In [22]:
Acapulco_user_icc_t_value_cc_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_CC", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_cc_vs_ai <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_CC", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### CC vs CA

In [23]:
Acapulco_user_icc_t_value_cc_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("statistic_CC", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_user_icc_d_value_cc_vs_ca <- icc(Acapulco_user_merged_full_folds_matchit[, c("effect.size_CC", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

### Merge ICCs

In [24]:
Acapulco_user_icc <- list(t_value_fullvsmatchit = Acapulco_user_icc_t_value_full_vs_matchit,
                        d_value_fullvsmatchit = Acapulco_user_icc_d_value_full_vs_matchit,
                        t_value_fullvsfolds = Acapulco_user_icc_t_value_full_vs_folds,
                        d_value_fullvsfolds = Acapulco_user_icc_d_value_full_vs_folds,
                        t_value_fullvscc = Acapulco_user_icc_t_value_full_vs_cc,
                        d_value_fullvscc = Acapulco_user_icc_d_value_full_vs_cc,
                        t_value_fullvsai = Acapulco_user_icc_t_value_full_vs_ai,
                        d_value_fullvsai = Acapulco_user_icc_d_value_full_vs_ai,
                        t_value_fullvsca = Acapulco_user_icc_t_value_full_vs_ca,
                        d_value_fullvsca = Acapulco_user_icc_d_value_full_vs_ca,
                        t_value_matchitvsfolds = Acapulco_user_icc_t_value_matchit_vs_folds,
                        d_value_matchitvsfolds = Acapulco_user_icc_d_value_matchit_vs_folds,
                        t_value_matchitvscc = Acapulco_user_icc_t_value_matchit_vs_cc,
                        d_value_matchitvscc = Acapulco_user_icc_d_value_matchit_vs_cc,
                        t_value_matchitvsai = Acapulco_user_icc_t_value_matchit_vs_ai,
                        d_value_matchitvsai = Acapulco_user_icc_d_value_matchit_vs_ai,
                        t_value_matchitvsca = Acapulco_user_icc_t_value_matchit_vs_ca,
                        d_value_matchitvsca = Acapulco_user_icc_d_value_matchit_vs_ca,
                        t_value_foldsvscc = Acapulco_user_icc_t_value_folds_vs_cc,
                        d_value_foldsvscc = Acapulco_user_icc_d_value_folds_vs_cc,
                        t_value_foldsvsai = Acapulco_user_icc_t_value_folds_vs_ai,
                        d_value_foldsvsai = Acapulco_user_icc_d_value_folds_vs_ai,
                        t_value_foldsvsca = Acapulco_user_icc_t_value_folds_vs_ca,
                        d_value_foldsvsca = Acapulco_user_icc_d_value_folds_vs_ca,
                        t_value_ccvsai = Acapulco_user_icc_t_value_cc_vs_ai,
                        d_value_ccvsai = Acapulco_user_icc_d_value_cc_vs_ai,
                        t_value_ccvsca = Acapulco_user_icc_t_value_cc_vs_ca,
                        d_value_ccvsca = Acapulco_user_icc_d_value_cc_vs_ca)

### **1.2 Substance**

final files:  
**Acapulco_substance_merged_full_folds_matchit**  
**Acapulco_substance_icc**

### Calculate

#### Full vs matchID

In [25]:
Acapulco_substance_icc_t_value_full_vs_matchit <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic", "statistic_matchit")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_full_vs_matchit <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_matchit")], model = "twoway", type = "agreement", unit = "single")

#### Full vs folds

In [26]:
Acapulco_substance_icc_t_value_full_vs_folds <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_full_vs_folds <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### Full vs CC

In [27]:
Acapulco_substance_icc_t_value_full_vs_cc <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic", "statistic_CC")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_full_vs_cc <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_CC")], model = "twoway", type = "agreement", unit = "single")

#### Full vs AI

In [28]:
Acapulco_substance_icc_t_value_full_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_full_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### Full vs CA

In [29]:
Acapulco_substance_icc_t_value_full_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_full_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs folds

In [30]:
Acapulco_substance_icc_t_value_matchit_vs_folds <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_matchit", "statistic_median")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_matchit_vs_folds <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_median")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs CC

In [31]:
Acapulco_substance_icc_t_value_matchit_vs_cc <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_matchit", "statistic_CC")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_matchit_vs_cc <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_CC")], model = "twoway", type = "agreement", unit = "single")  

#### matchID vs AI

In [32]:
Acapulco_substance_icc_t_value_matchit_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_matchit", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_matchit_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### matchID vs CA

In [34]:
Acapulco_substance_icc_t_value_matchit_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_matchit", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_matchit_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_matchit", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

#### folds vs CC

In [35]:
Acapulco_substance_icc_t_value_folds_vs_cc <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_median", "statistic_CC")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_folds_vs_cc <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_median", "effect.size_CC")], model = "twoway", type = "agreement", unit = "single")

#### folds vs AI

In [36]:
Acapulco_substance_icc_t_value_folds_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_median", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_folds_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_median", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### folds vs CA

In [38]:
Acapulco_substance_icc_t_value_folds_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_median", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_folds_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_median", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

#### CC vs AI

In [39]:
Acapulco_substance_icc_t_value_cc_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_CC", "statistic_AI")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_cc_vs_ai <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_CC", "effect.size_AI")], model = "twoway", type = "agreement", unit = "single")

#### CC vs CA

In [40]:
Acapulco_substance_icc_t_value_cc_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("statistic_CC", "statistic_CA")], model = "twoway", type = "agreement", unit = "single")
Acapulco_substance_icc_d_value_cc_vs_ca <- icc(Acapulco_substance_merged_full_folds_matchit[, c("effect.size_CC", "effect.size_CA")], model = "twoway", type = "agreement", unit = "single")

### Merge ICC

In [42]:
Acapulco_substance_icc <- list(t_value_fullvsmatchit = Acapulco_substance_icc_t_value_full_vs_matchit,
                        d_value_fullvsmatchit = Acapulco_substance_icc_d_value_full_vs_matchit,
                        t_value_fullvsfolds = Acapulco_substance_icc_t_value_full_vs_folds,
                        d_value_fullvsfolds = Acapulco_substance_icc_d_value_full_vs_folds,
                        t_value_fullvscc = Acapulco_substance_icc_t_value_full_vs_cc,
                        d_value_fullvscc = Acapulco_substance_icc_d_value_full_vs_cc,
                        t_value_fullvsai = Acapulco_substance_icc_t_value_full_vs_ai,
                        d_value_fullvsai = Acapulco_substance_icc_d_value_full_vs_ai,
                        t_value_fullvsca = Acapulco_substance_icc_t_value_full_vs_ca,
                        d_value_fullvsca = Acapulco_substance_icc_d_value_full_vs_ca,
                        t_value_matchitvsfolds = Acapulco_substance_icc_t_value_matchit_vs_folds,
                        d_value_matchitvsfolds = Acapulco_substance_icc_d_value_matchit_vs_folds,
                        t_value_matchitvscc = Acapulco_substance_icc_t_value_matchit_vs_cc,
                        d_value_matchitvscc = Acapulco_substance_icc_d_value_matchit_vs_cc,
                        t_value_matchitvsai = Acapulco_substance_icc_t_value_matchit_vs_ai,
                        d_value_matchitvsai = Acapulco_substance_icc_d_value_matchit_vs_ai,
                        t_value_matchitvsca = Acapulco_substance_icc_t_value_matchit_vs_ca,
                        d_value_matchitvsca = Acapulco_substance_icc_d_value_matchit_vs_ca,
                        t_value_foldsvscc = Acapulco_substance_icc_t_value_folds_vs_cc,
                        d_value_foldsvscc = Acapulco_substance_icc_d_value_folds_vs_cc,
                        t_value_foldsvsai = Acapulco_substance_icc_t_value_folds_vs_ai,
                        d_value_foldsvsai = Acapulco_substance_icc_d_value_folds_vs_ai,
                        t_value_foldsvsca = Acapulco_substance_icc_t_value_folds_vs_ca,
                        d_value_foldsvsca = Acapulco_substance_icc_d_value_folds_vs_ca,
                        t_value_ccvsai = Acapulco_substance_icc_t_value_cc_vs_ai,
                        d_value_ccvsai = Acapulco_substance_icc_d_value_cc_vs_ai,
                        t_value_ccvsca = Acapulco_substance_icc_t_value_cc_vs_ca,
                        d_value_ccvsca = Acapulco_substance_icc_d_value_cc_vs_ca)

### Plot ICC

#### Plot cohensD elements

In [43]:
Acapulco_all_merged_full_folds_matchit <- full_join(Acapulco_substance_merged_full_folds_matchit %>% select(region,Substance,contains("effect"),contains("statistic")),
    Acapulco_user_merged_full_folds_matchit %>% select(region,contains("effect"),contains("statistic")) %>% 
    mutate(Substance = "Users")) %>% mutate(Substance = factor(Substance, levels = c("Users", "ALC", "ATS", "CANN", "COC", "NIC", "OPI")))

plot_ICC_CohensD_fullvsmatchit <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size, y = effect.size_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_fullvsmatchit$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_fullvsmatchit$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_fullvsmatchit$ubound,3)),
        x = "Full sample",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_fullvsfolds <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size, y = effect.size_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_fullvsfolds$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_fullvsfolds$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_fullvsfolds$ubound,3)),
        x = "Full sample",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_fullvsCC <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size, y = effect.size_CC, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_fullvscc$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_fullvscc$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_fullvscc$ubound,3)),
        x = "Full sample",
        y = "Complete cases") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_fullvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size, y = effect.size_AI, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_fullvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_fullvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_fullvsai$ubound,3)),
        x = "Full sample",
        y = "Alt. imputation") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_fullvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size, y = effect.size_CA, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_fullvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_fullvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_fullvsca$ubound,3)),
        x = "Full sample",
        y = "Complete cases") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_matchitvsfolds <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_median, y = effect.size_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_matchitvsfolds$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_matchitvsfolds$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_matchitvsfolds$ubound,3)),
        x = "Median folds",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_matchitvsCC <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_CC, y = effect.size_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_matchitvscc$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_matchitvscc$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_matchitvscc$ubound,3)),
        x = "Complete cases",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_matchitvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_AI, y = effect.size_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_matchitvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_matchitvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_matchitvsai$ubound,3)),
        x = "Alt. imputation",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_matchitvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_CA, y = effect.size_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_matchitvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_matchitvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_matchitvsca$ubound,3)),
        x = "Complete cases",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_foldsvsCC <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_CC, y = effect.size_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_foldsvscc$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_foldsvscc$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_foldsvscc$ubound,3)),
        x = "Complete cases",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_foldsvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_AI, y = effect.size_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_foldsvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_foldsvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_foldsvsai$ubound,3)),
        x = "Alt. imputation",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_foldsvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_CA, y = effect.size_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_foldsvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_foldsvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_foldsvsca$ubound,3)),
        x = "Complete cases",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_CCsvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_AI, y = effect.size_CC, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_ccvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_ccvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_ccvsai$ubound,3)),
        x = "Alt. imputation",
        y = "Complete cases") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)

plot_ICC_CohensD_CCsvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = effect.size_CA, y = effect.size_CC, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() + 
    labs(title = paste0("Cohen's D: ICC = ",round(Acapulco_substance_icc$d_value_ccvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$d_value_ccvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$d_value_ccvsca$ubound,3)),
        x = "Complete cases",
        y = "Cov. Adjusted") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-0.4, 0.5) +
    ylim(-0.4, 0.5)


Joining with `by = join_by(region, Substance, effect.size, effect.size_median,
effect.size_mode, effect.size_media, effect.size_matchit,
diff_effect.size_std_FullvsMatchit, diff_effect.size_std_FullvsFolds,
effect.size_CC, effect.size_AI, effect.size_CA, statistic, statistic_median,
statistic_mode, statistic_media, statistic_matchit,
diff_statistic_abs_FullvsMatchit, diff_statistic_abs_FullvsFolds, statistic_CC,
statistic_AI, statistic_CA)`


#### Plot statistics

In [44]:

plot_ICC_statistic_fullvsmatchit <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic, y = statistic_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_fullvsmatchit$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_fullvsmatchit$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_fullvsmatchit$ubound,3)),
        x = "Full sample",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_fullvsfolds <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic, y = statistic_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_fullvsfolds$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_fullvsfolds$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_fullvsfolds$ubound,3)),
        x = "Full sample",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_fullvsCC <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic, y = statistic_CC, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_fullvscc$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_fullvscc$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_fullvscc$ubound,3)),
        x = "Full sample",
        y = "Complete cases") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_fullvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic, y = statistic_AI, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_fullvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_fullvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_fullvsai$ubound,3)),
        x = "Full sample",
        y = "Alt. imputation") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_fullvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic, y = statistic_CA, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_fullvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_fullvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_fullvsca$ubound,3)),
        x = "Full sample",
        y = "Cov. Adjusted") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_matchitvsfolds <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_median, y = statistic_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_matchitvsfolds$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_matchitvsfolds$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_matchitvsfolds$ubound,3)),
        x = "Median folds",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_matchitvsCC <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_CC, y = statistic_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_matchitvscc$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_matchitvscc$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_matchitvscc$ubound,3)),
        x = "Complete cases",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_matchitvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_AI, y = statistic_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_matchitvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_matchitvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_matchitvsai$ubound,3)),
        x = "Alt. imputation",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_matchitvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_CA, y = statistic_matchit, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_matchitvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_matchitvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_matchitvsca$ubound,3)),
        x = "Cov. Adjusted",
        y = "Matched sample") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_foldsvsCC <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_CC, y = statistic_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_foldsvscc$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_foldsvscc$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_foldsvscc$ubound,3)),
        x = "Complete cases",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_foldsvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_AI, y = statistic_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_foldsvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_foldsvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_foldsvsai$ubound,3)),
        x = "Alt. imputation",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_foldsvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_CA, y = statistic_median, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_foldsvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_foldsvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_foldsvsca$ubound,3)),
        x = "Cov. Adjusted",
        y = "Median folds") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_CCsvsAI <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_AI, y = statistic_CC, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_ccvsai$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_ccvsai$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_ccvsai$ubound,3)),
        x = "Alt. imputation",
        y = "Complete cases") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

plot_ICC_statistic_CCvsCA <- ggplot(Acapulco_all_merged_full_folds_matchit, aes(x = statistic_CA, y = statistic_CC, color = Substance)) +
    geom_point(alpha = 0.5) +
    geom_smooth(method = "lm", se = FALSE, size = 1.5) +
    ggpubr::stat_cor(
        aes(color = Substance),
        method = "pearson",
        show.legend = FALSE,
        label.x.npc = "left",
        label.y.npc = c(0.98, 0.93, 0.88, 0.83, 0.78, 0.73, 0.68),
        size = 5,
        geom = "label",
        fill = "white"
    ) +
    scale_color_manual(values = c("black", pal_substances)) +
    theme_pubr() +
    labs(title = paste0("Statistic: ICC = ",round(Acapulco_substance_icc$t_value_ccvsca$value,3)),
            subtitle = paste0("",
            round(Acapulco_substance_icc$t_value_ccvsca$lbound,3),
            " < ",
            "ICC",
            " < ",
            round(Acapulco_substance_icc$t_value_ccvsca$ubound,3)),
        x = "Cov. Adjusted",
        y = "Complete cases") +
    theme_output +
    theme(
        plot.title = element_text(hjust = 0.5,size=24),
        plot.subtitle = element_text(hjust = 0.5,size=23)) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey") +
    xlim(-8, 7) +
    ylim(-8, 7)

#### Merge figure

In [1]:
Figure_S27 <- plot_ICC_CohensD_fullvsmatchit + 
    plot_ICC_CohensD_fullvsfolds + 
    plot_ICC_CohensD_fullvsCC +
    plot_ICC_CohensD_fullvsAI +
    plot_ICC_CohensD_foldsvsCA +
    plot_ICC_CohensD_matchitvsfolds + 
    plot_ICC_CohensD_matchitvsCC +
    plot_ICC_CohensD_matchitvsAI +
    plot_ICC_CohensD_matchitvsCA +
    plot_ICC_CohensD_foldsvsCC +
    plot_ICC_CohensD_foldsvsAI +
    plot_ICC_CohensD_foldsvsCA +
    plot_ICC_CohensD_CCsvsAI +
    plot_ICC_CohensD_CCsvsCA +
    plot_ICC_statistic_fullvsmatchit + 
    plot_ICC_statistic_fullvsfolds +
    plot_ICC_statistic_fullvsCC +
    plot_ICC_statistic_fullvsAI +
    plot_ICC_statistic_fullvsCA +
    plot_ICC_statistic_matchitvsfolds + 
    plot_ICC_statistic_matchitvsCC +
    plot_ICC_statistic_matchitvsAI +
    plot_ICC_statistic_matchitvsCA +
    plot_ICC_statistic_foldsvsCC +
    plot_ICC_statistic_foldsvsAI +
    plot_ICC_statistic_foldsvsCA +
    plot_ICC_statistic_CCsvsAI +
    plot_ICC_statistic_CCvsCA +
    plot_layout(ncol=4, nrow=7, guides = "collect") &
    theme(plot.tag = element_text(size = 20, face = "bold"),
        legend.position = "bottom",
        legend.direction = "horizontal",
        legend.title = element_blank(),
        legend.text = element_text(size=25,margin=margin(0,18,0,3))
        ) &
    guides(color = guide_legend(nrow = 1, byrow = TRUE))

sc_supp_dir <- file.path("Figures", "Supplementaries")
dir.create(file.path(sc_supp_dir, "Figure_S27"), showWarnings = FALSE, recursive = TRUE)
out_S27 <- file.path(sc_supp_dir, "Figure_S27", "Figure_S27.png")
if (!file.exists(out_S27)) {
    ggsave(filename = out_S27, plot = Figure_S27, device = "png",
           width = 21, height = 24, units = "in", dpi = 300)
}

ERROR: Error: object 'plot_ICC_CohensD_fullvsmatchit' not found


## Save

In [46]:
Acapulco_user_merged_full_folds_matchit %>% write_csv("Statistics/Linear_Models/Acapulco/Acapulco_user_vs_control.csv")
Acapulco_substance_merged_full_folds_matchit %>% write_csv("Statistics/Linear_Models/Acapulco/Acapulco_substances_vs_control.csv")

In [47]:
save(Acapulco_user_merged_full_folds_matchit,
    Acapulco_user_icc,
    Acapulco_substance_merged_full_folds_matchit,
    Acapulco_substance_icc,
    file = "Statistics/Linear_Models/Acapulco/Acapulco_full_folds_matchit_comparison.RData")